<a href="https://colab.research.google.com/github/EllenEufrasio/Algoritmos_e_Logicas_de_Programacao/blob/main/Sess%C3%B5es_05_e_06_%E2%80%94_Exerc%C3%ADcios_(entrega).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nome: Ellen Eufrasio de Oliveira
RA: 1131392413021

# Sessões 05 e 06 — Exercícios (entrega)

**Processamento de Linguagem Natural**

Atividade prática

In [7]:
teste = 'ellen'

In [8]:
import re
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

---

## Desafio — comparando os três métodos numa base bem mais suja

Até agora usamos o HateBR, que já vem bem preparado (uma coluna de texto, um rótulo limpo). Bases do mundo real raramente chegam assim. Vamos usar o **B2W-Reviews01** — mais de 130 mil avaliações reais de produtos da Americanas.com, coletadas em 2018, com nota de 1 a 5 estrelas e se a pessoa recomendaria o produto a um amigo (CC BY-NC-SA 4.0).

Fonte: [github.com/b2wdigital/b2w-reviews01](https://github.com/b2wdigital/b2w-reviews01)

Essa base é genuinamente suja: tem linha sem texto, review duplicada, texto em CAIXA ALTA, preço colado no meio da frase ("R$1994.20,eu consegui..."), números, pontuação estranha. Antes de classificar qualquer coisa, precisa limpar.

In [14]:
from sklearn.model_selection import train_test_split

url_b2w = "https://raw.githubusercontent.com/b2wdigital/b2w-reviews01/master/B2W-Reviews01.csv"
b2w = pd.read_csv(url_b2w, low_memory=False)

print("Linhas:", len(b2w))
print("Colunas:", b2w.columns.tolist())
print()
print("Nulos em review_text:", b2w["review_text"].isna().sum())
print("Nulos em recommend_to_a_friend:", b2w["recommend_to_a_friend"].isna().sum())
print("Reviews duplicadas:", b2w["review_text"].duplicated().sum())
print()
print(b2w[["overall_rating", "recommend_to_a_friend", "review_text"]].sample(3, random_state=1).to_string())

Linhas: 132373
Colunas: ['submission_date', 'reviewer_id', 'product_id', 'product_name', 'product_brand', 'site_category_lv1', 'site_category_lv2', 'review_title', 'overall_rating', 'recommend_to_a_friend', 'review_text', 'reviewer_birth_year', 'reviewer_gender', 'reviewer_state']

Nulos em review_text: 3275
Nulos em recommend_to_a_friend: 18
Reviews duplicadas: 5648

       overall_rating recommend_to_a_friend                                                                                                                       review_text
9967                1                    No                                                                                                                               NaN
62225               3                   Yes  a churrasqueira veio com defeito, o furo do meio não está centralizado e tem uma borda menor que a outra, mesmo assim me atende.
40902               2                    No                                                  Como vou saber

In [10]:
#Comando
b2w

,submission_date,reviewer_id,product_id,product_name,product_brand,site_category_lv1,site_category_lv2,review_title,overall_rating,recommend_to_a_friend,review_text,reviewer_birth_year,reviewer_gender,reviewer_state
0,2018-01-01 00:11:28,d0fb1ca69422530334178f5c8624aa7a99da47907c44de...,132532965,Notebook Asus Vivobook Max X541NA-GO472T Intel...,NaN,Informática,Notebook,Bom,4,Yes,Estou contente com a compra entrega rápida o ú...,1958.0,F,RJ
1,2018-01-01 00:13:48,014d6dc5a10aed1ff1e6f349fb2b059a2d3de511c7538a...,22562178,Copo Acrílico Com Canudo 500ml Rocie,NaN,Utilidades Domésticas,"Copos, Taças e Canecas","Preço imbatível, ótima qualidade",4,Yes,"Por apenas R$1994.20,eu consegui comprar esse ...",1996.0,M,SC
2,2018-01-01 00:26:02,44f2c8edd93471926fff601274b8b2b5c4824e386ae4f2...,113022329,Panela de Pressão Elétrica Philips Walita Dail...,philips walita,Eletroportáteis,Panela Elétrica,ATENDE TODAS AS EXPECTATIVA.,4,Yes,SUPERA EM AGILIDADE E PRATICIDADE OUTRAS PANEL...,1984.0,M,SP
3,2018-01-01 00:35:54,ce741665c1764ab2d77539e18d0e4f66dde6213c9f0863...,113851581,Betoneira Columbus - Roma Brinquedos,roma jensen,Brinquedos,Veículos de Brinquedo,presente mais que desejado,4,Yes,MEU FILHO AMOU! PARECE DE VERDADE COM TANTOS D...,1985.0,F,SP
4,2018-01-01 01:00:28,7d7b6b18dda804a897359276cef0ca252f9932bf4b5c8e...,131788803,"Smart TV LED 43"" LG 43UJ6525 Ultra HD 4K com C...",lg,TV e Home Theater,TV,"Sem duvidas, excelente",5,Yes,"A entrega foi no prazo, as americanas estão de...",1994.0,M,MG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132368,2018-05-31 23:30:50,15f20e95ff44163f3175aaf67a5ae4a94d5030b409e521...,17962233,Carregador De Pilha Sony + 4 Pilhas Aa 2500mah,NaN,Câmeras e Filmadoras,Acessórios para Câmeras e Filmadoras,Ótimo produto!,5,Yes,"Vale muito, estou usando no controle do Xbox e...",1988.0,M,RS
132369,2018-05-31 23:42:25,def7cf9028b0673ab8bca3b1d06e085461fafb88cd48d9...,132631701,Mop Giratório Fit + Refil Extra - At Home,NaN,Utilidades Domésticas,Material de Limpeza,Sensacional,5,Yes,"Prático e barato, super indico o produto para ...",1979.0,F,SP
132370,2018-05-31 23:44:16,7bcbf542f5d7dd9a9a192a6805adba7a7a4c1ce3bf00df...,16095859,Fita Led 5m Rgb 3528 Siliconada Com 300 Leds C...,NaN,Automotivo,Iluminação,Ótimo produto,4,Yes,Chegou antes do prazo previsto e corresponde a...,1979.0,F,PR
132371,2018-05-31 23:46:48,e6fb0b19277d01c2a300c7837a105f3c369377e92f9c19...,6774907,Etiquetas Jurídicas Vade Mecum - Marca Fácil,marca facil,Papelaria,Material de Escritório,O produto não é bom.,1,No,"Material fraco, poderia ser melhor. Ficou deve...",1991.0,M,RJ


### Passo 1 — Limpar e preparar

O rótulo que vamos prever é `recommend_to_a_friend` ("Yes"/"No" — vira 1/0). Antes de vetorizar:

1. Tire as linhas com `review_text` ou `recommend_to_a_friend` nulos (`.dropna`).
2. Tire as reviews duplicadas (`.drop_duplicates(subset=["review_text"])`).
3. **(Sua vez)** Escreva uma função `limpar_review(texto)` usando regex que:
   - Converte pra minúsculas;
   - Troca valores em reais (`R$1994.20`, `r$ 50,00` etc.) por um marcador `" dinheiro "` (dica: `r'r\$\s*\d+[.,]?\d*'`);
   - Troca qualquer número restante por `" numero "`;
   - Remove tudo que não for letra (acentuada ou não) ou espaço;
   - Colapsa espaços duplos.
4. Aplique a função em toda a coluna `review_text` com `.apply(...)`.
5. Por que faz sentido trocar preço/número por um marcador em vez de simplesmente apagar? (Pense no que aconteceria com "R$1994.20,eu consegui comprar" se você só apagasse os dígitos sem colocar nada no lugar.)

Amostre **6.000 linhas** (`.sample(6000, random_state=1)`) antes de seguir — a base inteira é grande demais pra rodar tudo em segundos no Colab gratuito.

In [22]:
# Passo 1 - Limpar e preparar

#puxando os imports do inicio desse notebook
import re
import pandas as pd

#puxando a base de dados (data)
url_b2w = "https://raw.githubusercontent.com/b2wdigital/b2w-reviews01/master/B2W-Reviews01.csv"
b2w = pd.read_csv(url_b2w, low_memory=False)

# 1 e 2. Remover nulos e duplicatas
#1. Tire as linhas com review_text ou recommend_to_a_friend nulos (.dropna).
#2. Tire as reviews duplicadas (.drop_duplicates(subset=["review_text"])).


b2w_limpo = b2w.dropna(subset=["review_text", "recommend_to_a_friend"]).drop_duplicates(subset=["review_text"].copy())


# Amostre 6.000 linhas (.sample(6000, random_state=1)) antes de seguir — a base inteira é grande demais pra rodar tudo em segundos no Colab gratuito.
b2w_limpo = b2w_limpo.sample(6000, random_state=1).copy()

# 3. Função de limpeza com regex
#3. (Sua vez) Escreva uma função limpar_review(texto) usando regex que:


# anotação de uma solução usada: A função isinstance em Python é usada para verificar se um objeto é de um tipo específico ou pertence a uma classe específica. Ela retorna True se o objeto for do tipo especificado, caso contrário, retorna False.
def limpar_review(texto):
    if not isinstance(texto, str):
        return ""

    # Converter para minúsculas
    texto = texto.lower()

    # Troca valores em reais (R$1994.20, r$ 50,00 etc.) por um marcador " dinheiro " (dica: r'r\$\s*\d+[.,]?\d*');
    texto = re.sub(r'r\\\$\s*\\d+[.,]?\\d*', ' dinheiro ', texto)

    # Troca qualquer número restante por " numero ";
    texto = re.sub(r'\\d+', ' numero ', texto)

    # Remove tudo que não for letra (acentuada ou não) ou espaço;
    texto = re.sub(r'[^a-zA-Záéíóúâêîôûãõç\s]', ' ', texto)

    # Colapsa espaços duplos.
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto

# 4. Aplicar a função de limpeza
b2w_limpo["texto_limpo"] = b2w_limpo["review_text"].apply(limpar_review)

print("Quantidade de avaliações:", len(b2w_limpo))
print(b2w_limpo[["review_text", "texto_limpo", "recommend_to_a_friend"]].head())

#outra maneira de exibir os resultados
display(b2w_limpo[["review_text", "texto_limpo"]].head())

display(b2w_limpo[["recommend_to_a_friend", "texto_limpo"]].head())



Quantidade de avaliações: 6000
                                              review_text  \
107174  Não gostei,  Muito limitado  Não reproduz as m...   
117314  O produto é maravilhoso, resistente, de grande...   
104742  A qualidade superou as expectativas, e realmen...   
94465   Produto muito bom varias opções de corte preci...   
50087   Comprei esse kit de 10 lâmpadas, só que ao inv...   

                                              texto_limpo  \
107174  não gostei muito limitado não reproduz as mídi...   
117314  o produto é maravilhoso resistente de grande q...   
104742  a qualidade superou as expectativas e realment...   
94465   produto muito bom varias opções de corte preci...   
50087   comprei esse kit de lâmpadas só que ao invés d...   

       recommend_to_a_friend  
107174                    No  
117314                   Yes  
104742                   Yes  
94465                    Yes  
50087                     No  


,review_text,texto_limpo
107174,"Não gostei, Muito limitado Não reproduz as m...",não gostei muito limitado não reproduz as mídi...
117314,"O produto é maravilhoso, resistente, de grande...",o produto é maravilhoso resistente de grande q...
104742,"A qualidade superou as expectativas, e realmen...",a qualidade superou as expectativas e realment...
94465,Produto muito bom varias opções de corte preci...,produto muito bom varias opções de corte preci...
50087,"Comprei esse kit de 10 lâmpadas, só que ao inv...",comprei esse kit de lâmpadas só que ao invés d...


,recommend_to_a_friend,texto_limpo
107174,No,não gostei muito limitado não reproduz as mídi...
117314,Yes,o produto é maravilhoso resistente de grande q...
104742,Yes,a qualidade superou as expectativas e realment...
94465,Yes,produto muito bom varias opções de corte preci...
50087,No,comprei esse kit de lâmpadas só que ao invés d...


# 5. Resposta teórica:
Faz sentido trocar preço/número por um marcador em vez de simplesmente apagar, porque
se apenas apagar os numeros "1994.20" da frase "RS1994.20,eu consegui", o texto viraria "R$,eu consegui", logo a frase ficaria com sentido incompleto.

Sem o espaço e sem a separação correta, as palavras poderiam se colar ("reuconsegui"),
prejudicando a identificação e a tokenização. Além disso, manter os marcadores de "dinheiro" e de "numero" preserva a informação semântica de que o usuário mencionou valores ou números, o que é útil para o modelo.



### Passo 2 — Os três métodos, na mesma base

Agora repita a comparação de hoje, só que nesta base nova:

1. **Detector por palavra-chave** (como `contem_linguagem_inadequada` das Sessões 03-04): monte uma lista pequena de palavras claramente negativas ("péssimo", "horrível", "não recomendo"...) e classifique como "não recomenda" (0) qualquer review que contenha alguma delas; o resto vira "recomenda" (1).
2. **Léxico de sentimento** (`polaridade_frase_simples` + `lexico`, já carregados no setup): some a polaridade da review; se a soma for negativa, classifique como "não recomenda".
3. **Classificador TF-IDF + Naive Bayes**: repita o loop de unigrama / uni+bigrama / uni+bi+trigrama que já fizemos no material, agora com `X_treino`/`X_teste` vindos de `texto_limpo`.

Pra cada um, calcule a acurácia contra `recommend_to_a_friend` (convertido pra 1/0) e monte uma tabela comparando os três, igual à do material de hoje.

**Antes de comparar:** calcule também a acurácia de um "modelo" que sempre chuta a classe mais frequente (`y_treino.mode()[0]` repetido pra todo mundo). Essa base **não é balanceada** como o HateBR — vale a pena saber qual é o piso antes de comemorar qualquer acurácia.

In [48]:
# Passo 2

# 2.1. Preparando os dados (importando novamente, o que já foi declardo lá em cima)
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

y = (b2w_limpo["recommend_to_a_friend"] == "Yes").astype(int)
X = b2w_limpo["texto_limpo"]

# divisão de treino e para teste
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)

resultados = {}

# 2.1a. Baseline "sempre a classe majoritária"
classe_majoritaria = y_treino.mode()[0]

# Chutando sempre a classe majoritária para todo o conjunto de teste
y_pred_baseline = [classe_majoritaria] * len(y_teste)
acuracia_baseline = accuracy_score(y_teste, y_pred_baseline)
resultados["Baseline (Majoritária)"] = acuracia_baseline

print(f"A classe majoritária é: {classe_majoritaria} ('Yes' se 1, 'No' se 0)")
print(f"Acurácia do Baseline: {acuracia_baseline:.4f}")

A classe majoritária é: 1 ('Yes' se 1, 'No' se 0)
Acurácia do Baseline: 0.7350


In [49]:
# 2.1b. Detector por palavra-chave
# usei IA para gerar n palavras, queria testar com muitas palavras e prevendo o problema de colocar numeros no meio das palavras
palavras_negativas = palavras_negativas = [
    # Palavras já existentes
    "ruim", "ruim demais", "ruim d+", "ruim de mais", "ru1m",
    "p3ssimo", "pessim0", "pessima", "péssima", "p3ss1m0",
    "pe3ss1mo", "pessimo", "péssimo", "horrivel", "horrível",
    "odiei", "defeito", "lixo", "fraco", "quebrou", "estragou",

    # Qualidade ruim
    "péssimos", "péssimas", "pessimos", "pessimas",
    "horrível", "horriveis", "horríveis", "terrível",
    "terrivel", "terríveis", "terriveis", "medíocre",
    "mediocre", "inferior", "inadequado", "inadequada",
    "defeituoso", "defeituosa", "malfeito", "mal feita",
    "malfeito demais", "qualidade ruim", "baixa qualidade",
    "má qualidade", "pouca qualidade", "qualidade péssima",

    # Insatisfação
    "odiei", "odeio", "detestei", "detesto", "não gostei",
    "nao gostei", "não gosto", "nao gosto", "insatisfeito",
    "insatisfeita", "decepcionado", "decepcionada",
    "decepção", "decepcao", "frustrante", "frustração",
    "frustracao", "arrependido", "arrependida",
    "me arrependi", "não recomendo", "nao recomendo",
    "não compraria novamente", "nunca mais compro",

    # Problemas e defeitos
    "defeito", "defeitos", "quebrou", "quebrado", "quebrada",
    "estragou", "estragado", "estragada", "danificado",
    "danificada", "avariado", "avariada", "rachado",
    "rachada", "descosturou", "descolou", "soltou",
    "não funciona", "nao funciona", "parou de funcionar",
    "parou de funcionar rápido", "veio com defeito",
    "veio quebrado", "veio estragado", "produto com defeito",
    "falha", "falhou", "defeituoso", "defeituosa",

    # Aparência e conservação
    "feio", "feia", "horrendo", "horrenda", "amassado",
    "amassada", "sujo", "suja", "manchado", "manchada",
    "velho", "velha", "mal conservado", "mal conservada",
    "acabamento ruim", "aparência ruim", "parece falsificado",

    # Atendimento e experiência
    "péssimo atendimento", "pessimo atendimento",
    "atendimento ruim", "mal atendido", "mal atendida",
    "desrespeito", "desrespeitoso", "desrespeitosa",
    "demorado", "demorada", "atrasado", "atrasada",
    "atraso", "não entregaram", "nao entregaram",
    "não recebi", "nao recebi", "entrega atrasada",
    "pedido errado", "produto errado", "veio errado",
    "propaganda enganosa", "enganoso", "enganosa",

    # Preço e custo-benefício
    "caro demais", "cara demais", "muito caro",
    "muito cara", "preço abusivo", "preco abusivo",
    "não vale a pena", "nao vale a pena",
    "não vale o preço", "nao vale o preco",
    "dinheiro jogado fora", "joguei dinheiro fora",
    "desperdício de dinheiro", "desperdicio de dinheiro",

    # Gírias e expressões informais
    "lixo", "uma porcaria", "porcaria", "bosta",
    "horrível d+", "ruim pra caramba", "ruim pra cacete",
    "muito ruim", "ruim demais da conta", "péssimo d+",
    "pessimo d+", "fraco demais", "bem ruim",
    "nada bom", "nada boa", "uma droga", "uma merda",
    "sem condições", "sem condicoes", "não presta",
    "nao presta", "não gostei nadinha", "que horror",
    "vergonhoso", "vergonhosa", "ridículo", "ridicula",
    "ridícula", "inútil", "inutil",

    # Abreviações e variações digitais
    "ruim d+", "ruim d mais", "ru1m d+", "r u i m",
    "pessim0", "p3ssimo", "p3ssima", "p3ss1m0",
    "p3ss1ma", "h0rrivel", "h0rr1vel", "horr1vel",
    "l1xo", "l1x0", "od13i", "od3io", "d3feito",
    "qu3brou", "estrag0u", "nao g0stei", "n gostei",
    "n recomendo", "n vale a pena", "n presta",
    "péssimooo", "pessimoooo", "ruimmmm",
    "horriveeeel", "lixooo", "porcariaaa",

    # Reclamações sobre desempenho
    "lento", "lenta", "muito lento", "muito lenta",
    "travando", "trava muito", "não carrega",
    "nao carrega", "demora muito", "instável",
    "instavel", "complicado", "complicada",
    "difícil de usar", "dificil de usar",
    "não abre", "nao abre", "erro constante",
    "cheio de erros", "cheia de erros",
    "funcionamento ruim", "desempenho ruim"
]

def prever_por_palavra_chave(texto):
    # Se contiver alguma palavra negativa, classificamos como 0 (não recomenda)
    for palavra in palavras_negativas:
        if palavra in texto:
            return 0
    return 1 # Caso contrário, recomenda

y_pred_palavras = X_teste.apply(prever_por_palavra_chave)
acuracia_palavras = accuracy_score(y_teste, y_pred_palavras)
resultados["Detector Palavra-Chave"] = acuracia_palavras

print(f"Acurácia do Detector por Palavra-Chave: {acuracia_palavras:.4f}")


Acurácia do Detector por Palavra-Chave: 0.8342


In [50]:
# 2.2. Léxico de sentimento simples
# dicionário de polaridades
lexico_sentimento = {
    "otimo": 2, "ótimo": 2, "bom": 1, "excelente": 2, "maravilhoso": 2, "recomendo": 2, "gostei": 1, "amei": 2, "perfeito": 2,
    "ruim": -1, "pessimo": -2, "péssimo": -2, "horrivel": -2, "horrível": -2, "odiei": -2, "defeito": -1, "lixo": -2, "fraco": -1, "quebrou": -1
}

def calcular_sentimento_lexico(texto):
    score = 0
    palavras = texto.split()
    for palavra in palavras:
        if palavra in lexico_sentimento:
            score += lexico_sentimento[palavra]
    # Se a soma total for negativa, classifica como 0, caso contrário 1
    return 0 if score < 0 else 1

y_pred_lexico = X_teste.apply(calcular_sentimento_lexico)
acuracia_lexico = accuracy_score(y_teste, y_pred_lexico)
resultados["Léxico de Sentimento"] = acuracia_lexico

print(f"Acurácia do Léxico de Sentimento: {acuracia_lexico:.4f}")

Acurácia do Léxico de Sentimento: 0.7708


In [51]:
# 2.3. TF-IDF + Naive Bayes
configuracoes = {
    "TF-IDF + NB (Unigramas)": (1, 1),
    "TF-IDF + NB (Uni+Bigramas)": (1, 2),
    "TF-IDF + NB (Uni+Bi+Trigramas)": (1, 3)
}

for nome, ngram_range in configuracoes.items():
    vetorizador = TfidfVectorizer(ngram_range=ngram_range)

    X_treino_vet = vetorizador.fit_transform(X_treino)
    X_teste_vet = vetorizador.transform(X_teste)

    clf = MultinomialNB()
    clf.fit(X_treino_vet, y_treino)

    y_pred = clf.predict(X_teste_vet)
    acuracia = accuracy_score(y_teste, y_pred)

    resultados[nome] = acuracia
    print(f"Acurácia do {nome}: {acuracia:.4f}")

Acurácia do TF-IDF + NB (Unigramas): 0.8233
Acurácia do TF-IDF + NB (Uni+Bigramas): 0.7875
Acurácia do TF-IDF + NB (Uni+Bi+Trigramas): 0.7767


In [53]:
# Tabela comparativa
df_comparativo = pd.DataFrame(list(resultados.items()), columns=["Método", "Acurácia"])
display(df_comparativo.sort_values(by="Acurácia", ascending=False).reset_index(drop=True))

,Método,Acurácia
0,Detector Palavra-Chave,0.834167
1,TF-IDF + NB (Unigramas),0.823333
2,TF-IDF + NB (Uni+Bigramas),0.787500
3,TF-IDF + NB (Uni+Bi+Trigramas),0.776667
4,Léxico de Sentimento,0.770833
5,Baseline (Majoritária),0.735000


### Passo 3 — Reflexão

Responda com um comentário no código (não precisa de resposta "certa", mas precisa ser sua conclusão de verdade, olhando os números que você achou):

1. Qual dos três métodos ficou mais perto do teto (ML) e qual ficou mais perto do piso (baseline sempre-a-mesma-classe)?

**Resposta:**
Nas minhas análises, o que se saiu melhor foi *(tirando o detector por Palavras-Chave, que eu adicionei muitas possíveis palavras)* o TF-IDF + Naive Bayes (Unigramas) com **82,33%** (como mostrado na tabelinha acima).

Já o com menor acurária, foi o método Léxico de Sentimento, que obteve apenas **77,08%** de acurácia, sendo uma diferença bem significativa.

2. Essa base tem bem mais reviews positivas (recomenda) que negativas. Isso muda como você interpreta "86% de acurácia", por exemplo? Uma acurácia alta aqui significa a mesma coisa que uma acurácia alta no HateBR (que é balanceado 50/50)?

**Resposta:**
Sim, muda completamente a interpretação, porque em uma base balanceada de 50/50 (como a base do HateBR), o acerto por puro palpite ou baseline é de 50%, então 86% de acurácia indica um ganho massivo de performance (saldo positivo de 36%). Nesta base do B2W, chutar apenas que "Sim" já garante uma acurácia de **73,50%**. Portanto, uma acurácia de 86% aqui representa um ganho real menor sobre o acaso, exigindo métricas complementares (como F1-score) para avaliar a qualidade real das predições nas classes minoritárias.

3. Se você tivesse que escolher só **um** desses três métodos pra colocar no ar amanhã, num sistema real da loja, qual escolheria — e o que pesaria mais na decisão além da acurácia (custo computacional, interpretabilidade, precisa de dado rotulado ou não)?

**Resposta:**
Eu escolheria o **TF-IDF + Naive Bayes (Unigramas)**, por ser mais prático, porque por exemplo, o detector de palavras-chave obteve uma  acurácia significativamente superior, mas isso foi devido à lista enorme de termos manuais (mesmo que utilize uma IA para gerar essa lista, ainda sim é um processo mais "manual"), manter e atualizar essa lista manualmente no mundo real para capturar gírias novas e erros de digitação, possíveis maneiras de burlar, como é o caso de utilizar números, caracter especial no meio é inviável e pouco escalável.

Já com o Naive Bayes é extremamente leve computacionalmente falando, ele tem uma alta interpretabilidade de uma maneira mais "expert", é possível inspecionar as probabilidades das palavras e aprender as nuances do vocabulário automaticamente a partir de dados rotulados.